Phenyo Thato Molete- 216038155
BLCH9X2 Assignment 2:

In [14]:
from copy import deepcopy

from block import (
    Block,
    GENESIS_PREVIOUS_HASH,
    canonical_dumps,
    create_genesis_block,
    create_linked_block,
    is_hash_valid,
    sha256_string,)


In [15]:
def main() -> None:
    print("=" * 60)
    print("A2 — Block Construction Lab demo")
    print("=" * 60)

#Build a genesis block and show its material fields 
genesis = create_genesis_block(timestamp=1_700_000_000)
print("\n[1] Genesis block")
for key, value in genesis.to_dict().items():
    print(f"    {key}: {value}")
assert genesis.index == 0
assert genesis.previous_hash == GENESIS_PREVIOUS_HASH == "0" * 64
assert genesis.hash == genesis.compute_hash()



[1] Genesis block
    index: 0
    timestamp: 1700000000
    transactions: []
    previous_hash: 0000000000000000000000000000000000000000000000000000000000000000
    nonce: 0
    hash: 6b5ab8051e3b3b82c560233df8634f71d8debba962f8e2d28b898c012d5ff2f1


In [16]:
# Show the exact canonicalisation used for the commitment 
raw = canonical_dumps(genesis.payload_for_hash())
print("\n[2] Canonical JSON fed into the hash")
print(f"    {raw}")
print(f"    sha256(...) = {sha256_string(raw)}")
assert sha256_string(raw) == genesis.hash



[2] Canonical JSON fed into the hash
    {"index":0,"nonce":0,"previous_hash":"0000000000000000000000000000000000000000000000000000000000000000","timestamp":1700000000,"transactions":[]}
    sha256(...) = 6b5ab8051e3b3b82c560233df8634f71d8debba962f8e2d28b898c012d5ff2f1


In [17]:
# (c) Link a second block
sample_tx = [{"sender": "Alice", "recipient": "Bob", "amount": 10}]
block1 = create_linked_block(genesis, sample_tx, timestamp=1_700_000_100)
print("\n[3] Block 1 (linked to genesis)")
for key, value in block1.to_dict().items():
    print(f"    {key}: {value}")
print(f"    previous_hash == genesis.hash ? {block1.previous_hash == genesis.hash}")
assert block1.previous_hash == genesis.hash
assert is_hash_valid(block1)


[3] Block 1 (linked to genesis)
    index: 1
    timestamp: 1700000100
    transactions: [{'sender': 'Alice', 'recipient': 'Bob', 'amount': 10}]
    previous_hash: 6b5ab8051e3b3b82c560233df8634f71d8debba962f8e2d28b898c012d5ff2f1
    nonce: 0
    hash: e3e436fbd939f7d7094277fa2b1cd5b3d68dee86e58956606c82cac6d61a6b94
    previous_hash == genesis.hash ? True


In [18]:
# (d) Tamper evidence demo 
print("\n[4] Tamper evidence demo")
tampered = deepcopy(block1)
stored_before = tampered.hash
print(f"    stored hash before tamper : {stored_before}")

tampered.transactions[0]["amount"] = 999  # silent post-hoc edit
recomputed = tampered.compute_hash()
print(f"    recomputed after amount=999 (unchanged stored hash): {recomputed}")
print(f"    stored == recomputed?      {stored_before == recomputed}")
print(f"    is_hash_valid(tampered)?   {is_hash_valid(tampered)}")

assert stored_before != recomputed
assert is_hash_valid(tampered) is False
# original block1 must be untouched (deepcopy protected it)
assert is_hash_valid(block1) is True
assert block1.transactions[0]["amount"] == 10
print("\nAll checks passed: genesis built, block linked, tamper detected.")



[4] Tamper evidence demo
    stored hash before tamper : e3e436fbd939f7d7094277fa2b1cd5b3d68dee86e58956606c82cac6d61a6b94
    recomputed after amount=999 (unchanged stored hash): 0e6f0e3d584f47549b03172669b030741d33606fdbeb1adc0f5eb1dec7320328
    stored == recomputed?      False
    is_hash_valid(tampered)?   False

All checks passed: genesis built, block linked, tamper detected.
